In [0]:
file_list = dbutils.widgets.get("file_list")
rs_oblist_path = dbutils.widgets.get("rs_oblist_path")
landing_table = dbutils.widgets.get("landing_table")
landing_history_table = dbutils.widgets.get("landing_history_table")

In [0]:
%skip
file_list = ['oblist011126.xlsx']
rs_oblist_path ="/Volumes/dev_bronze_landing/alphacollector/source_files/rs_oblist/"
landing_table = "dev_silver_lakehouse.staging.bears_oblist" 
landing_history_table = "dev_silver_lakehouse.staging.bears_oblist_history" 

In [0]:
spark.sql(f"""
TRUNCATE TABLE {landing_table};
""")

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd
from io import BytesIO
import ast
file_list = ast.literal_eval(file_list)

for file_name in file_list:
    xlsx_file_path = rs_oblist_path + file_name

    binary_df = spark.read.format("binaryFile").load(xlsx_file_path)
    file_content = binary_df.collect()[0]['content']
    pdf = pd.read_excel(BytesIO(file_content), sheet_name=0, engine='openpyxl')
    print(f"✓ Read Excel: {len(pdf)} rows, {len(pdf.columns)} columns")
        
    record_count = len(pdf)
    pdf = pdf.replace({pd.NA: None, pd.NaT: None, float('nan'): None, None: None}).fillna('')
    pdf = pdf.astype(str)
    rs_oblist_file = spark.createDataFrame(pdf)
    rs_oblist_file.createOrReplaceTempView("rs_oblist_view")

    # load into staginng table
    spark.sql(f"""
    INSERT INTO {landing_table}
    SELECT
        COALESCE(try_CAST(Office AS INT), 0) AS office,
        OABB AS oabb,
        CLIENTNO AS client_no,
        CLIENTNAME AS client_name,
        CLIENTFIRSTNAME AS client_first_name,
        INVNO AS inv_no,
        COALESCE(try_CAST(INVDATE AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS inv_date,
        PAYORNAME AS payor_name,
        COALESCE(try_CAST(ORIGBILL AS DOUBLE), 0.0) AS orig_bill,
        try_cast(try_CAST(`GROUP#` AS DOUBLE) AS INT )  AS group_id,
        `billing team3` AS billing_team3,
        try_cast(try_CAST(TEAM AS DOUBLE) AS INT) AS team,
        `payor type code` AS payor_type_code,
        PAYORTYPE AS payor_type,
        BILLTO AS bill_to,
        `BILL TO PROGRAM` AS bill_to_program,
        COALESCE(try_CAST(TOTALDUE AS DOUBLE), 0.0) AS total_due,
        COALESCE(try_CAST(Current AS DOUBLE), 0.0) AS current_amt,
        COALESCE(try_CAST(`4 - 7 WKS` AS DOUBLE), 0.0) AS weeks_4_7,
        COALESCE(try_CAST(`8 - 13 WKS` AS DOUBLE), 0.0) AS weeks_8_13,
        COALESCE(try_CAST(`14 wks to Reserve` AS DOUBLE), 0.0) AS weeks_14_to_reserve,
        COALESCE(try_CAST(`Up for Reserve` AS DOUBLE), 0.0) AS up_for_reserve,
        COALESCE(try_CAST(`Prior Reserve` AS DOUBLE), 0.0) AS prior_reserve,
        COALESCE(try_CAST(NET AS DOUBLE), 0.0) AS net_amt,
        Specialty AS specialty,
        `PAYOR CATEGORY` AS payor_category,
        `CLIENT STATE` AS client_state,
        `COLLECTOR NAME` AS collector_name,
        COALESCE(try_CAST(`last post date` AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS last_post_date,
        `client status` AS client_status,
        company AS company,
        COALESCE(try_CAST(`last payment date` AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS last_payment_date,
        COALESCE(try_CAST(`Serv W/E Date` AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS service_week_end_date,
        NULL AS reporting_week_ending_date,
        "{file_name[:-5]}" AS _file_name,
        current_timestamp() AS _load_timestamp
    FROM rs_oblist_view;
    """)

    # load into history table
    spark.sql(f"""
    INSERT INTO {landing_history_table}
    SELECT
        COALESCE(try_CAST(Office AS INT), 0) AS office,
        OABB AS oabb,
        CLIENTNO AS client_no,
        CLIENTNAME AS client_name,
        CLIENTFIRSTNAME AS client_first_name,
        INVNO AS inv_no,
        COALESCE(try_CAST(INVDATE AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS inv_date,
        PAYORNAME AS payor_name,
        COALESCE(try_CAST(ORIGBILL AS DOUBLE), 0.0) AS orig_bill,
        try_cast(try_CAST(`GROUP#` AS DOUBLE) AS INT )  AS group_id,
        `billing team3` AS billing_team3,
        try_cast(try_CAST(TEAM AS DOUBLE) AS INT) AS team,
        `payor type code` AS payor_type_code,
        PAYORTYPE AS payor_type,
        BILLTO AS bill_to,
        `BILL TO PROGRAM` AS bill_to_program,
        COALESCE(try_CAST(TOTALDUE AS DOUBLE), 0.0) AS total_due,
        COALESCE(try_CAST(Current AS DOUBLE), 0.0) AS current_amt,
        COALESCE(try_CAST(`4 - 7 WKS` AS DOUBLE), 0.0) AS weeks_4_7,
        COALESCE(try_CAST(`8 - 13 WKS` AS DOUBLE), 0.0) AS weeks_8_13,
        COALESCE(try_CAST(`14 wks to Reserve` AS DOUBLE), 0.0) AS weeks_14_to_reserve,
        COALESCE(try_CAST(`Up for Reserve` AS DOUBLE), 0.0) AS up_for_reserve,
        COALESCE(try_CAST(`Prior Reserve` AS DOUBLE), 0.0) AS prior_reserve,
        COALESCE(try_CAST(NET AS DOUBLE), 0.0) AS net_amt,
        Specialty AS specialty,
        `PAYOR CATEGORY` AS payor_category,
        `CLIENT STATE` AS client_state,
        `COLLECTOR NAME` AS collector_name,
        COALESCE(try_CAST(`last post date` AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS last_post_date,
        `client status` AS client_status,
        company AS company,
        COALESCE(try_CAST(`last payment date` AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS last_payment_date,
        COALESCE(try_CAST(`Serv W/E Date` AS TIMESTAMP), try_CAST(NULL AS TIMESTAMP)) AS service_week_end_date,
        NULL AS reporting_period,
        "{file_name[:-5]}" AS _file_name,
        current_timestamp() AS _load_timestamp
    FROM rs_oblist_view;
    """)

In [0]:
spark.sql(f"""
UPDATE {landing_table}
SET reporting_week_ending_date =
    date_add(to_date(RIGHT(_file_name, 6), 'MMddyy'), 1 - dayofweek(to_date(RIGHT(_file_name, 6), 'MMddyy')))
""")

In [0]:
spark.sql(f"""
UPDATE {landing_history_table}
SET reporting_period =
    date_add(to_date(RIGHT(_file_name, 6), 'MMddyy'), 1 - dayofweek(to_date(RIGHT(_file_name, 6), 'MMddyy')))
WHERE reporting_period IS NULL
""")